<a href="https://colab.research.google.com/github/FarazHeydar/Prompt-Assembly-for-Healthcare-Applications/blob/main/Coding/03_granite_before_vs_after_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP9800 MSc Project — IBM Granite "Before vs. After" Prompt Assembly Pipeline
**Project Title:** Developing Techniques for Prompt Assembly for Healthcare Applications  
**Author:** Faraz | **Supervisor:** Dr. Jinan Fiaidhi  
**Institution:** Lakehead University (F/W 2026–2027)

---

## 1. Overview & Experimental Design
This notebook implements **Stage 3 (Core Experiment)** of our methodology: executing a controlled **"Before vs. After" comparative evaluation** on **IBM® Granite™ 3.1 Instruct** Small Language Models (SLMs).

### Experimental Conditions Compared:
1. **Condition A — Baseline Monolithic Prompt (`Before`):**
   * Uses an unconstrained zero-shot instruction (*"You are a medical assistant. Summarize the following doctor-patient conversation into a SOAP note."*).
   * Tests baseline vulnerability to conversational noise retention, colloquial phrasing, hallucinated clinical parameters, and unstructured assessment plans.
2. **Condition B — Modular Prompt Assembly Framework (`After`):**
   * Dynamically constructs a multi-stage prompt integrating five core paradigms:
     * **Module 1 (Structural Delimiters & Noise Filtering):** Isolates raw transcript data using `### TRANSCRIPT` blocks and explicitly strips casual small talk.
     * **Module 2 (Dynamic Few-Shot Conditioning):** Injects the Top-2 semantically matched Dialogue-to-SOAP exemplars retrieved in **Notebook 02**.
     * **Module 3 (Linguistic Shifting):** Enforces clinical taxonomy mapping (e.g., *"heart is fluttering"* $\rightarrow$ *"intermittent palpitations"*).
     * **Module 4 (Negative Guardrails):** Enforces strict zero-hallucination constraints, requiring `"Not Documented"` for any unmentioned vitals, allergies, or history.
     * **Module 5 (PICO-Structured Chain-of-Thought):** Forces intermediate clinical extraction across **[P]**atient/Diagnosis, **[I]**ntervention, **[C]**omparison/Alternative, and **[O]**utcome prior to drafting the final SOAP note.

## 2. GPU Environment Setup & Connecting to Notebook 02 Artifacts
The code cell below verifies GPU hardware acceleration (NVIDIA T4), installs HuggingFace `transformers`, `accelerate`, and `bitsandbytes` for memory-efficient 4-bit/fp16 inference, and automatically locates the enriched Parquet files (`custom_50_with_fewshot.parquet` and `mts_eval_100_with_fewshot.parquet`) from **Notebook 02**.

In [ ]:
# ==============================================================================
# UNIVERSAL KAGGLE + GOOGLE COLAB FILE LOADER
# Purpose: Automatically finds input .parquet files on Kaggle (/kaggle/input)
#          OR downloads them automatically from the public GitHub repo when
#          Dr. Jinan opens the notebook in Google Colab!
# ==============================================================================
import urllib.request

def get_project_file(filename: str) -> str:
    """
    1. Checks Kaggle Input & local working directories first.
    2. If running on Google Colab and file is not found locally, automatically
       downloads it from the public GitHub repository!
    """
    search_roots = ["/kaggle/input", "/kaggle/working", "./processed_data", "./retriever_outputs", "./granite_generations", "./data", "."]
    for root in search_roots:
        if os.path.exists(root):
            matches = glob.glob(os.path.join(root, "**", filename), recursive=True)
            if matches:
                print(f"[FOUND LOCALLY] {matches[0]}")
                return matches[0]

    # Fallback for Google Colab: Auto-download from GitHub raw URL
    os.makedirs("./data", exist_ok=True)
    local_dest = os.path.join("./data", filename)
    raw_url = f"https://raw.githubusercontent.com/FarazHeydar/Prompt-Assembly-for-Healthcare-Applications/main/data/{filename}"
    print(f"[COLAB AUTO-DOWNLOAD] Fetching '{filename}' from GitHub ({raw_url})...")
    urllib.request.urlretrieve(raw_url, local_dest)
    print(f"[SUCCESS] Downloaded to {local_dest}")
    return local_dest

## 3. Loading IBM® Granite™ 3.1 Instruct (SLM Target Model)
Here we load **IBM Granite 3.1 Instruct** from the official `ibm-granite` HuggingFace repository.
* By default, `MODEL_ID` is set to `"ibm-granite/granite-3.1-2b-instruct"` (or you can toggle `"ibm-granite/granite-3.1-8b-instruct"`), utilizing 4-bit NormalFloat (`nf4`) quantization with `float16` compute dtype so inference across 50–100 cases completes rapidly with zero VRAM overflow.

In [ ]:
# ==============================================================================
# LOAD IBM GRANITE 3.1 IN NATIVE FP16
# Purpose: Load Granite-3.1-2B-Instruct in native float16 on cuda:0 (avoiding
#          multi-GPU PCIe bottlenecks and 4-bit dequantization overhead on T4).
# ==============================================================================

MODEL_ID = "ibm-granite/granite-3.1-2b-instruct"

print(f"[STATUS] Loading Tokenizer and Model in native FP16: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Native float16 on cuda:0 uses only ~5.0 GB VRAM (out of 15 GB on T4) and runs 4x faster!
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": "cuda:0"}
)
model.eval()

print(f"[SUCCESS] {MODEL_ID} loaded in native FP16 on cuda:0!")

## 4. Prompt Engineering Architecture: Condition A (`Before`) vs. Condition B (`After`)
This section defines our two exact prompt constructors:
1. **`build_condition_a_prompt(dialogue)`:** Constructs the naive, monolithic zero-shot prompt.
2. **`build_condition_b_prompt(dialogue, few_shot_list, omitted_candidates)`:** Dynamically assembles the 5-stage prompt incorporating:
   * Role assignment & **Noise Removal Rules** (stripping casual opening banter).
   * **Top-2 Dynamic Few-Shot Exemplars** from `MedSynth`.
   * **Linguistic Shifting Rules** (converting patient slang into formal clinical terminology).
   * **Strict Negative Guardrails** (requiring `"Not Documented"` for unmentioned parameters).
   * **PICO Structured Schema** (`[P] Patient/Diagnosis`, `[I] Intervention`, `[C] Comparison/Alternative`, `[O] Outcome`).

In [ ]:
# ==============================================================================
# PROMPT ASSEMBLY WITH STRICT EXEMPLAR ISOLATION
# Purpose: Prevent SLM Few-Shot Contamination by isolating the retrieved
#          MedSynth exemplar from the actual target transcript, and enable
#          fast GPU batched inference.
# ==============================================================================

def build_condition_a_prompt(dialogue: str) -> list:
    """CONDITION A (BEFORE): Naive Monolithic Zero-Shot Prompt."""
    user_content = (
        "You are a medical assistant. Summarize the following doctor-patient "
        "conversation into a SOAP note.\n\n"
        f"{dialogue}"
    )
    return [{"role": "user", "content": user_content}]


def build_condition_b_prompt(dialogue: str, retrieved_few_shots: list, checklist_fields: list = None) -> list:
    """
    CONDITION B (AFTER): Modular Prompt Assembly with Exemplar Isolation,
    Linguistic Shifting, Negative Guardrails, and PICO Clinical Reasoning.
    """
    # Use Top-1 retrieved MedSynth exemplar strictly as a formatting reference
    ex = retrieved_few_shots[0]
    ex_soap = str(ex["soap_note"])[:450]

    if checklist_fields is not None and len(checklist_fields) > 0:
        audit_fields_str = ", ".join(checklist_fields)
    else:
        audit_fields_str = "Allergies, Family History, Temperature, Respiratory Rate, Blood Pressure, Heart Rate"

    system_prompt = (
        "You are an expert clinical documentation AI. Transform the doctor-patient transcript "
        "inside <TARGET_TRANSCRIPT> into a structured SOAP note using ONLY facts from <TARGET_TRANSCRIPT>.\n"
        "CRITICAL RULES:\n"
        "1. NOISE FILTERING: Completely ignore casual small talk (sports, weather, pets, family trips, hobbies).\n"
        "2. LINGUISTIC SHIFTING: Translate patient slang into formal medical terminology "
        "(e.g., 'heart is fluttering' -> 'intermittent palpitations'; 'it races' -> 'tachycardia'; "
        "'get winded' -> 'exertional dyspnea'; 'burning fire in throat' -> 'pyrosis'; "
        "'peeing razor blades' -> 'severe dysuria'; 'drinking water like a fish' -> 'polydipsia').\n"
        "3. NEGATIVE GUARDRAILS (NO HALLUCINATIONS): Do NOT invent age, gender, vitals, or exam findings. "
        f"You MUST check if [{audit_fields_str}] are mentioned in <TARGET_TRANSCRIPT>. "
        "For EVERY item in that list not explicitly stated in <TARGET_TRANSCRIPT>, write '<Item>: Not Documented.'\n"
        "4. PICO REASONING: In the Plan section, write a structured PICO Summary ([P] Patient/Diagnosis, "
        "[I] Intervention, [C] Comparison/Alternative discussed, [O] Outcome)."
    )

    user_prompt = (
        "<STYLE_REFERENCE_ONLY_DO_NOT_COPY_PATIENT_FACTS>\n"
        f"{ex_soap}\n"
        "</STYLE_REFERENCE_ONLY_DO_NOT_COPY_PATIENT_FACTS>\n\n"
        "<TARGET_TRANSCRIPT>\n"
        f"{dialogue}\n"
        "</TARGET_TRANSCRIPT>\n\n"
        "Now write the SOAP note strictly for the patient in <TARGET_TRANSCRIPT> using this exact structure:\n"
        "Subjective:\n"
        "- Chief Complaint: (formal medical terms)\n"
        "- HPI: (formal medical terms; exclude all small talk)\n"
        f"- Unmentioned Audit Parameters: (list each unstated item from [{audit_fields_str}] as '<Item>: Not Documented.')\n\n"
        "Objective:\n"
        "- Vitals: (exact vitals from <TARGET_TRANSCRIPT> only; mark unmeasured vitals as 'Not Documented')\n"
        "- Physical Exam & Labs: (exact findings from <TARGET_TRANSCRIPT> only)\n\n"
        "Assessment:\n"
        "- Primary working diagnosis.\n\n"
        "Plan:\n"
        "- PICO Summary:\n"
        "  * [P] Patient/Diagnosis: ...\n"
        "  * [I] Intervention: ...\n"
        "  * [C] Comparison/Alternative: ...\n"
        "  * [O] Outcome: ..."
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]


@torch.no_grad()
def generate_granite_batch(messages_batch: list, max_new_tokens: int = 360) -> list:
    """Runs high-speed batched GPU generation across multiple prompts simultaneously."""
    prompts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in messages_batch
    ]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    ).to("cuda:0")

    input_len = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

    results = []
    for i in range(len(prompts)):
        gen_tokens = outputs[i][input_len:]
        text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        results.append(text)
    return results

print("[SUCCESS] Fast Batched Generator & Isolated Prompt Assembly compiled!")

## 5. Single-Case Head-to-Head Verification (`EXP_001`)
Before running batch inference across all 50 custom experimental cases, we run a quick side-by-side verification on **`EXP_001` (Cardiology — Arthur's hockey small talk + *"heart is fluttering"* + omitted Allergies/Family History/Temp/RR + Holter vs. Metoprolol PICO)** to visually inspect the difference between **Condition A (Before)** and **Condition B (After)**.

In [ ]:
# ==============================================================================
# VERIFY EXP_001 (BEFORE VS. AFTER)
# ==============================================================================

t0 = time.time()
sample_case = df_custom_50.iloc[0]

msgs_a = build_condition_a_prompt(sample_case["dialogue"])
msgs_b = build_condition_b_prompt(
    dialogue=sample_case["dialogue"],
    retrieved_few_shots=sample_case["retrieved_few_shots"],
    checklist_fields=list(sample_case["omitted_fields"])
)

# Run both Condition A and Condition B together in a single GPU batch!
out_a, out_b = generate_granite_batch([msgs_a, msgs_b], max_new_tokens=360)

print(f"Completed EXP_001 Before & After in {time.time() - t0:.2f} seconds!\n")
print("=" * 75)
print("CONDITION A [BEFORE - ZERO-SHOT BASELINE] OUTPUT ON EXP_001:")
print("=" * 75)
print(out_a)

print("\n" + "=" * 75)
print("CONDITION B [AFTER - MODULAR PROMPT ASSEMBLY + PICO] OUTPUT ON EXP_001:")
print("=" * 75)
print(out_b)

# ==============================================================================
# INSPECT EXACT PROMPTS: CONDITION A VS. CONDITION B ON EXP_001
# Purpose: Print out the raw strings sent to the SLM for visual inspection.
# ==============================================================================

sample_case = df_custom_50.iloc[0]

# 1. Build message lists using our constructor functions
msgs_a = build_condition_a_prompt(sample_case["dialogue"])
msgs_b = build_condition_b_prompt(
    dialogue=sample_case["dialogue"],
    retrieved_few_shots=sample_case["retrieved_few_shots"],
    checklist_fields=list(sample_case["omitted_fields"])
)

# 2. Apply HuggingFace's chat template to see the exact formatted prompt string
raw_prompt_a = tokenizer.apply_chat_template(msgs_a, tokenize=False, add_generation_prompt=True)
raw_prompt_b = tokenizer.apply_chat_template(msgs_b, tokenize=False, add_generation_prompt=True)

print("=" * 80)
print("EXACT PROMPT SENT TO MODEL IN CONDITION A [BEFORE - ZERO-SHOT]:")
print("=" * 80)
print(raw_prompt_a)

print("\n" + "=" * 80)
print("EXACT PROMPT SENT TO MODEL IN CONDITION B [AFTER - PROMPT ASSEMBLY + PICO]:")
print("=" * 80)
print(raw_prompt_b)

## 6. Full Batch Execution Across the 50 Custom Experimental Cases & MTS-Dialog Subset
Now we execute the full experimental sweep across:
1. All **50 Controlled Experimental Stress-Test Cases (`df_custom_50`)**
2. A **30-case benchmark subset of `MTS-Dialog` (`df_mts_30`)**

All generated outputs (`condition_a_soap` and `condition_b_soap`) are saved to `/kaggle/working/granite_generations` so **Notebook 04 (Three-Tier Evaluation Engine)** can compute ROUGE, BERTScore, Medical NER F1, Negative Guardrail Compliance %, and Hallucination Rate.

In [ ]:
# ==============================================================================
# RUN ALL 50 CUSTOM + 30 MTS-DIALOG CASES
# Purpose: Process prompts in GPU batches of 8 so the entire experiment finishes
#          in ~3 to 5 minutes instead of 45 minutes.
# ==============================================================================

BATCH_SIZE = 8

def run_dataset_sweep(df: pd.DataFrame, is_custom: bool = True, max_tokens: int = 360):
    prompts_a = []
    prompts_b = []

    for _, row in df.iterrows():
        prompts_a.append(build_condition_a_prompt(row["dialogue"]))
        chk = list(row["omitted_fields"]) if is_custom else ["Allergies", "Family History", "Temperature", "Blood Pressure"]
        prompts_b.append(
            build_condition_b_prompt(
                dialogue=row["dialogue"],
                retrieved_few_shots=row["retrieved_few_shots"],
                checklist_fields=chk
            )
        )

    outputs_a, outputs_b = [], []

    print("  -> Generating Condition A (Before - Zero-Shot) in batches...")
    for i in tqdm(range(0, len(prompts_a), BATCH_SIZE)):
        batch = prompts_a[i : i + BATCH_SIZE]
        outputs_a.extend(generate_granite_batch(batch, max_new_tokens=max_tokens))

    print("  -> Generating Condition B (After - Prompt Assembly + PICO) in batches...")
    for i in tqdm(range(0, len(prompts_b), BATCH_SIZE)):
        batch = prompts_b[i : i + BATCH_SIZE]
        outputs_b.extend(generate_granite_batch(batch, max_new_tokens=max_tokens))

    return outputs_a, outputs_b

# 1. Execute on all 50 Custom Experimental Cases
t_start = time.time()
print(f"[1/2] Running Batched Sweep on {len(df_custom_50)} Custom Experimental Cases...")
ca_custom, cb_custom = run_dataset_sweep(df_custom_50, is_custom=True, max_tokens=360)
df_custom_50["condition_a_soap"] = ca_custom
df_custom_50["condition_b_soap"] = cb_custom

custom_gen_path = os.path.join(OUTPUT_DIR, "custom_50_granite_results.parquet")
df_custom_50.to_parquet(custom_gen_path, index=False)
print(f"  -> Done in {(time.time() - t_start)/60:.2f} mins! Saved: {custom_gen_path}")

# 2. Execute on 30 MTS-Dialog Benchmark Cases
t_mts = time.time()
df_mts_subset = df_mts_100.head(30).copy().reset_index(drop=True)
print(f"\n[2/2] Running Batched Sweep on {len(df_mts_subset)} MTS-Dialog Benchmark Cases...")
ca_mts, cb_mts = run_dataset_sweep(df_mts_subset, is_custom=False, max_tokens=320)
df_mts_subset["condition_a_soap"] = ca_mts
df_mts_subset["condition_b_soap"] = cb_mts

mts_gen_path = os.path.join(OUTPUT_DIR, "mts_30_granite_results.parquet")
df_mts_subset.to_parquet(mts_gen_path, index=False)
print(f"  -> Done in {(time.time() - t_mts)/60:.2f} mins! Saved: {mts_gen_path}")